## Serial stuff

In [1]:
!pip install pyserial

In [2]:
import serial, time
!pip install pyserial

In [3]:
#ser.close()

**Note:** if importing `serial` causes an error, you need to install the `pyserial` module using `pip`:

`pip install pyserial`

or 

`pip3 install pyserial`

Windows users should use the Anaconda prompt.  Mac users should be able to use the terminal.

In [4]:
print(serial)

<module 'serial' from 'C:\\Users\\boome\\anaconda3\\Lib\\site-packages\\serial\\__init__.py'>


In [5]:
print(serial.__file__)

C:\Users\boome\anaconda3\Lib\site-packages\serial\__init__.py


In [6]:
print(serial.__version__)

3.5


In [7]:
serial.VERSION

'3.5'

**Note:** if you serial version is 2.x, we might need to make changes to the code below

In [8]:
baudrate = 115200

In [9]:
#portname = '/dev/cu.usbmodem11301'#mac
portname = 'COM4'#windows

In [10]:
ser = serial.Serial(portname, baudrate, timeout=5)

In [11]:
ser.in_waiting

0

In [12]:
def read_all(ser):
    out = []
    while ser.in_waiting > 0:
        data1b = ser.read(1)
        data1 = data1b.decode('utf-8')
        out.append(data1)
        
    outstr = ''.join(out)
    return outstr

In [13]:
read_all(ser)

''

In [14]:
def read_one_line(ser):
    out = []
    while ser.in_waiting > 0:
        data1b = ser.read(1)
        data1 = data1b.decode('utf-8')
        if data1 in ['\n','\r']:
            break
        out.append(data1)
        
    outstr = ''.join(out)
    return outstr

In [15]:
read_one_line(ser)

'dual servo control over serial'

In [16]:
read_all(ser)

''

In [17]:
def one_byte_int_to_serial_byte(int_byte):
    out_byte = int(int_byte).to_bytes(1, byteorder='big')
    return out_byte

In [18]:
def WriteByte(ser, bytein):
    out_byte = one_byte_int_to_serial_byte(bytein)
    ser.write(out_byte)

In [19]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from numpy import sin, cos, tan, pi
import robotics
from robotics import Rx, Ry, Rz, sind, cosd, DH, prettymat
rtd = 180/pi
dtr = pi/180

## Parameters
![Screenshot 2026-06-19 234816.png](attachment:21bb921d-4bfa-4f88-9196-fb7a3789d1a5.png)

In [308]:
# measure based off sketch parameters
# units of cm
A = 11.5  # height of base of l1
B = 3     # offset from z1 to l1|
C = 5.5   # offset from l1 to l2
D = 1.5   # offset from l2 to center of EOAT
l1 = 24   # base link
l2 = 21.5 # tip link
l3 = 21   # gripper link

gripper_open = 1500
gripper_closed = 1050

################ for reference
#T01 = DH(0,0,th1+90,A)
#T12 = DH(90,0,th2+90,B)
#T23 = DH(180,l1,th3+90,C)
#T34 = DH(180,l2,th4-90,-D)

## Functions

In [309]:
# takes 3d coordinant and outputs required servo angles
def inverseKinematics(X, Y, Z):

    #define wrist position 
    P_wrist_0 = np.array([X,Y,Z+l3,1]) 
    
    #offset between base and wrist
    offset = B-C-D

    #planer distance from offset to wrist
    #(magnitude of distance from origin to tip) - (magnitude of offset)
    dist = np.sqrt(P_wrist_0[0]**2 + P_wrist_0[1]**2 - offset**2)

    #th1 = (angle made from base to wrist) +/- (angle made from dist and offset)
    #th1 = np.arctan2(P_wrist_0[1],P_wrist_0[0]) - np.arctan2(dist, offset)
    #below combines above to one arctan2 to avoid quadrant bug. +90 per DH table
    th1 = -np.arctan2(-dist*P_wrist_0[0] + offset*P_wrist_0[1], offset*P_wrist_0[0] + dist*P_wrist_0[1])*rtd + 90

    #distance from origin1 to wrist
    r_squared = dist**2 + (P_wrist_0[2]-A)**2
    
    #law of cos for angle between links. solved for both elbow up and down configs
    alpha_temp = (r_squared-l1**2-l2**2)/(-2*l1*l2)
    sin_alpha_p = np.sqrt(1-alpha_temp**2)
    sin_alpha_n = -sin_alpha_p
    alpha = np.arctan2(sin_alpha_p, alpha_temp)
    alpha_2 = np.arctan2(sin_alpha_n, alpha_temp)
    
    #vertical angle theorem for theta 2
    theta3 = 180 - alpha*rtd
    theta3_2 = 180 - alpha_2*rtd
    
    #triangle in link1 co-ordinant system for psi
    psi = np.arctan2(l2*sind(theta3), l1+l2*cosd(theta3))
    psi_2 = np.arctan2(l2*sind(theta3_2), l1+l2*cosd(theta3_2))
    
    #angle of r to x-axis
    beta = np.arctan2(P_wrist_0[2]-A, dist)
    
    #difference in beta and psi is theta 1
    theta2 = (beta + psi)*rtd
    theta2_2 = (beta + psi_2)*rtd

    #elbow UP config
    th2 = theta2
    th3 = theta3 

    #elbow DOWN congig
    #th2 = theta2_2
    #th3 = theta3_2 

    # l3 should be perpendicular to the ground at all times 
    # mechanical bug requires offset
    th4 = th3 - th2 + 90

    return th1, th2, th3, th4

In [311]:
def thetaInterpolation(th1, th2, th3, th4):
    #convert angle into arduino code
    theta_min = 0     # minimum angle
    theta_max = 180   # maximum angle
    min_new = 1000    # minimum servo value
    max_new = 2000    # maximum servo value

    #linear interpolate for the first theta value
    #min_new and max_new flipped 
    myint = max_new + ((th1-theta_min)*(min_new-max_new))/(theta_max-theta_min)

    #linear interpolate for the second theta value
    myint2 = min_new + ((th2-theta_min)*(max_new-min_new))/(theta_max-theta_min)

    #third theta (need to adjust once tested)
    myint3 = min_new + ((th3-theta_min)*(max_new-min_new))/(theta_max-theta_min)

    #fourth theta (need to adjust once tested)
    #plus 100 for mechanical bug
    myint4 = min_new + ((th4-theta_min)*(max_new-min_new))/(theta_max-theta_min) + 100

    return myint, myint2, myint3, myint4

In [282]:
def break_into_two(breakint):
    MSB = breakint // 256
    LSB = breakint % 256
    return MSB, LSB

## Test single position

In [313]:
test_xyz = np.array([B-C-D,l2,A+l1-l3]) #home position
test_xyz

array([-4. , 21.5, 14.5])

In [314]:
test_angles = inverseKinematics(test_xyz[0],test_xyz[1],test_xyz[2])
test_angles

(np.float64(90.0), np.float64(90.0), np.float64(90.0), np.float64(90.0))

In [316]:
################ for reference
#T01 = DH(0,0,th1+90,A)
#T12 = DH(90,0,th2+90,B)
#T23 = DH(180,l1,th3+90,C)
#T34 = DH(180,l2,th4-90,-D)

# rotations done to theta in original DH table are adjusted in invK function
T01 = DH(0,0,180-test_angles[0],A) # +90 done in invK and also comes out reversed (practically adjusted for in thetaInterp)
T12 = DH(90,0,test_angles[1],B) # +90 done in invK
T23 = DH(180,l1,test_angles[2],C) # +90 done in invK
T34 = DH(180,l2,test_angles[3]-90-90,-D) # -90 for reasons?

T04 = T01@T12@T23@T34
P_tip_4 = np.array([l3,0,0,1])

P_tip_0_check = T04 @ P_tip_4
prettymat(P_tip_0_check)

array([-4. , 21.5, 14.5,  1. ])

In [317]:
test_servo = thetaInterpolation(test_angles[0],test_angles[1],test_angles[2],test_angles[3])
test_servo

(np.float64(1500.0),
 np.float64(1500.0),
 np.float64(1500.0),
 np.float64(1600.0))

In [248]:
byte1_test, byte2_test = break_into_two(test_servo[0])
byte3_test, byte4_test = break_into_two(test_servo[1])
byte5_test, byte6_test = break_into_two(test_servo[2])
byte7_test, byte8_test = break_into_two(test_servo[3])
byte9_test, byte10_test = break_into_two(gripper_closed)

In [249]:
#send all test bytes to arduino
WriteByte(ser, int(byte1_test))   # servo 1 MSB
time.sleep(0.05)
WriteByte(ser, int(byte2_test))   # servo 1 LSB
time.sleep(0.05)
WriteByte(ser, int(byte3_test))   # servo 2 MSB
time.sleep(0.05)
WriteByte(ser, int(byte4_test))   # servo 2 LSB
time.sleep(0.05)
WriteByte(ser, int(byte5_test))   # servo 3 LSB
time.sleep(0.05)
WriteByte(ser, int(byte6_test))   # servo 3 LSB
time.sleep(0.05)
WriteByte(ser, int(byte7_test))   # servo 4 LSB
time.sleep(0.05)
WriteByte(ser, int(byte8_test))   # servo 4 LSB
time.sleep(0.05)
WriteByte(ser, int(byte9_test))   # servo 5 LSB
time.sleep(0.05)
WriteByte(ser, int(byte10_test))   # servo 5 LSB
time.sleep(0.05)

## Path Setup

In [318]:
# home position
homeX = B-C-D
homeY = l2
homeZ = A+l1-l3

#define pick location
pickX = 11.5
pickY = 21
pickZ = 3*2.54

#define place location
placeX = -15.5
placeY = 25
placeZ = 2*2.54

#define obstical location
obstacleX = 0
obstacleY = 20

lift = 5  # z displacement from pick
obstacle_radius = 2 * 2.45  # 2 inches in cm
clearance = 3 # buffer distance from obstacle (cm)

#number of positions for each path
N0 = 2
N1 = 3  # z  down into the pick
N2 = 2  # z  up above the place z
N3 = 2  # y  to below the obstacle
N4 = 10  # x  across to the place x
N5 = 3  # y  to the place y
N6 = 3  # z  down to the place
N7 = 4  # z  retract up 
N8 = 3  # return home

z_clear = max(pickZ, placeZ) + lift                          # travel height: above place z (and above pick)
y_safe  = obstacleY - obstacle_radius - clearance            # a lane in front of (below) the obstacle

path0 = np.linspace([homeX,   homeY,  homeZ],  [pickX, pickY, z_clear],   N0) # from home to above pick (to avoid obstacle)
path1 = np.linspace([pickX, pickY, z_clear],   [pickX, pickY, pickZ],     N1) # z  down into the pick
# ---- PAUSE: close gripper ----
path2 = np.linspace([pickX,  pickY,  pickZ],   [pickX,  pickY,  z_clear], N2)  # z  up above the place z
path3 = np.linspace([pickX,  pickY,  z_clear], [pickX,  y_safe, z_clear], N3)  # y  to below the obstacle
path4 = np.linspace([pickX,  y_safe, z_clear], [placeX, y_safe, z_clear], N4)  # x  across to the place x
path5 = np.linspace([placeX, y_safe, z_clear], [placeX, placeY, z_clear], N5)  # y  to the place y
path6 = np.linspace([placeX, placeY, z_clear], [placeX, placeY, placeZ],  N6)  # z  down to the place
# ---- PAUSE: open gripper ----
path7 = np.linspace([placeX, placeY, placeZ],  [placeX, placeY, z_clear], N7)  # z  retract up 
path8 = np.linspace([placeX, placeY, z_clear], [homeX,  homeY,  homeZ],   N8)  # return to home

path0

array([[-4.  , 21.5 , 14.5 ],
       [11.5 , 21.  , 12.62]])

In [319]:
# run each path through inverseKinemnatics function to get angles for each position
path0_angles = np.array([inverseKinematics(X, Y, Z) for X, Y, Z in path0])
path1_angles = np.array([inverseKinematics(X, Y, Z) for X, Y, Z in path1])
path2_angles = np.array([inverseKinematics(X, Y, Z) for X, Y, Z in path2])
path3_angles = np.array([inverseKinematics(X, Y, Z) for X, Y, Z in path3])
path4_angles = np.array([inverseKinematics(X, Y, Z) for X, Y, Z in path4])
path5_angles = np.array([inverseKinematics(X, Y, Z) for X, Y, Z in path5])
path6_angles = np.array([inverseKinematics(X, Y, Z) for X, Y, Z in path6])
path7_angles = np.array([inverseKinematics(X, Y, Z) for X, Y, Z in path7])
path8_angles = np.array([inverseKinematics(X, Y, Z) for X, Y, Z in path8])

path0_angles

array([[ 90.        ,  90.        ,  90.        ,  90.        ],
       [128.32322367,  84.78840473,  89.53949687,  94.75109214]])

In [304]:
# run each path_angles through thetaInterpolation function to get angles for each position in servo language
path0_servo = np.array([thetaInterpolation(th1, th2, th3, th4) for th1, th2, th3, th4 in path0_angles])
path1_servo = np.array([thetaInterpolation(th1, th2, th3, th4) for th1, th2, th3, th4 in path1_angles])
path2_servo = np.array([thetaInterpolation(th1, th2, th3, th4) for th1, th2, th3, th4 in path2_angles])
path3_servo = np.array([thetaInterpolation(th1, th2, th3, th4) for th1, th2, th3, th4 in path3_angles])
path4_servo = np.array([thetaInterpolation(th1, th2, th3, th4) for th1, th2, th3, th4 in path4_angles])
path5_servo = np.array([thetaInterpolation(th1, th2, th3, th4) for th1, th2, th3, th4 in path5_angles])
path6_servo = np.array([thetaInterpolation(th1, th2, th3, th4) for th1, th2, th3, th4 in path6_angles])
path7_servo = np.array([thetaInterpolation(th1, th2, th3, th4) for th1, th2, th3, th4 in path7_angles])
path8_servo = np.array([thetaInterpolation(th1, th2, th3, th4) for th1, th2, th3, th4 in path8_angles])

path0_servo

array([[1500.        , 1500.        , 1500.        , 1583.33333333],
       [1287.09320185, 1449.03241318, 1320.83179444, 1455.13271459]])

In [253]:
# add in gripper control
path0_full = np.column_stack((path0_servo, np.full(len(path0),gripper_open))) ## open gripper
path1_full = np.column_stack((path1_servo, np.full(len(path1),gripper_open))) ## open gripper
path2_full = np.column_stack((path2_servo, np.full(len(path2),gripper_closed))) ## closed gripper
path3_full = np.column_stack((path3_servo, np.full(len(path3),gripper_closed))) ## closed gripper
path4_full = np.column_stack((path4_servo, np.full(len(path4),gripper_closed))) ## closed gripper
path5_full = np.column_stack((path5_servo, np.full(len(path5),gripper_closed))) ## closed gripper
path6_full = np.column_stack((path6_servo, np.full(len(path6),gripper_closed))) ## closed gripper
path7_full = np.column_stack((path7_servo, np.full(len(path7),gripper_open))) ## open gripper
path8_full = np.column_stack((path8_servo, np.full(len(path8),gripper_open))) ## open gripper

path0_full

array([[1500.        , 1500.        , 1500.        , 1583.33333333,
        1500.        ],
       [1287.09320185, 1449.03241318, 1320.83179444, 1455.13271459,
        1500.        ]])

## Individual paths

### path 0: over pick

In [254]:
#define arrays to hold the four bytes
byte1_0 = np.zeros(len(path0_full), dtype=int)
byte2_0 = np.zeros(len(path0_full), dtype=int)
byte3_0 = np.zeros(len(path0_full), dtype=int)
byte4_0 = np.zeros(len(path0_full), dtype=int)
byte5_0 = np.zeros(len(path0_full), dtype=int)
byte6_0 = np.zeros(len(path0_full), dtype=int)
byte7_0 = np.zeros(len(path0_full), dtype=int)
byte8_0 = np.zeros(len(path0_full), dtype=int)
byte9_0 = np.zeros(len(path0_full), dtype=int)
byte10_0 = np.zeros(len(path0_full), dtype=int)

#store bytes into temp values to be sent to arduino
for i in range(len(path0_full)):
    byte1_0[i], byte2_0[i] = break_into_two(path0_full[i,0])
    byte3_0[i], byte4_0[i] = break_into_two(path0_full[i,1])
    byte5_0[i], byte6_0[i] = break_into_two(path0_full[i,2])
    byte7_0[i], byte8_0[i] = break_into_two(path0_full[i,3])
    byte9_0[i], byte10_0[i] = break_into_two(path0_full[i,4])

print(byte1_0,'\n\n',byte2_0,'\n\n\n',
      byte3_0,'\n\n',byte4_0,'\n\n\n',
      byte5_0,'\n\n',byte6_0,'\n\n\n',
      byte7_0,'\n\n',byte8_0,'\n\n\n',
      byte9_0,'\n\n',byte10_0)

[5 5] 

 [220   7] 


 [5 5] 

 [220 169] 


 [5 5] 

 [220  40] 


 [6 5] 

 [ 47 175] 


 [5 5] 

 [220 220]


In [255]:
# Send all path points to both servos
x = 1
for i in range(len(path0_full)):
    WriteByte(ser, int(byte1_0[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2_0[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3_0[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4_0[i]))   # servo 2 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte5_0[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte6_0[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte7_0[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte8_0[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte9_0[i]))   # servo 5 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte10_0[i]))   # servo 5 LSB
    time.sleep(0.5)

    print(f"Step {i:2d}:  "
          f"th1={byte1_0[i]*256 + byte2_0[i]}  "
          f"th2={byte3_0[i]*256 + byte4_0[i]}  "
          f"th3={byte5_0[i]*256 + byte6_0[i]}  "
          f"th4={byte7_0[i]*256 + byte8_0[i]}  "
          f"th5={byte9_0[i]*256 + byte10_0[i]}")
    
    if x == 1:
        x = 2
        
    else:
        
        while 1==1:
            response = ser.readline().decode('utf-8').strip()
            if response == "Ready":
                break
        x = 1
    #time.sleep(1)  # pause between steps so servo has time to move

Step  0:  th1=1500  th2=1500  th3=1500  th4=1583  th5=1500
Step  1:  th1=1287  th2=1449  th3=1320  th4=1455  th5=1500


### path 1: pick

In [256]:
#define arrays to hold the four bytes
byte1_1 = np.zeros(len(path1_full), dtype=int)
byte2_1 = np.zeros(len(path1_full), dtype=int)
byte3_1 = np.zeros(len(path1_full), dtype=int)
byte4_1 = np.zeros(len(path1_full), dtype=int)
byte5_1 = np.zeros(len(path1_full), dtype=int)
byte6_1 = np.zeros(len(path1_full), dtype=int)
byte7_1 = np.zeros(len(path1_full), dtype=int)
byte8_1 = np.zeros(len(path1_full), dtype=int)
byte9_1 = np.zeros(len(path1_full), dtype=int)
byte10_1 = np.zeros(len(path1_full), dtype=int)

#store bytes into temp values to be sent to arduino
for i in range(len(path1_full)):
    byte1_1[i], byte2_1[i] = break_into_two(path1_full[i,0])
    byte3_1[i], byte4_1[i] = break_into_two(path1_full[i,1])
    byte5_1[i], byte6_1[i] = break_into_two(path1_full[i,2])
    byte7_1[i], byte8_1[i] = break_into_two(path1_full[i,3])
    byte9_1[i], byte10_1[i] = break_into_two(path1_full[i,4])

print(byte1_1,'\n\n',byte2_1,'\n\n\n',
      byte3_1,'\n\n',byte4_1,'\n\n\n',
      byte5_1,'\n\n',byte6_1,'\n\n\n',
      byte7_1,'\n\n',byte8_1,'\n\n\n',
      byte9_1,'\n\n',byte10_1)

[5 5 5] 

 [7 7 7] 


 [5 5 5] 

 [169 191 177] 


 [5 5 6] 

 [ 40 181  22] 


 [5 6 6] 

 [175  36 147] 


 [5 5 5] 

 [220 220 220]


In [257]:
# Send all path points to both servos
x = 1
for i in range(len(path1_full)):
    WriteByte(ser, int(byte1_1[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2_1[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3_1[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4_1[i]))   # servo 2 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte5_1[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte6_1[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte7_1[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte8_1[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte9_1[i]))   # servo 5 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte10_1[i]))   # servo 5 LSB
    time.sleep(0.5)

    print(f"Step {i:2d}:  "
          f"th1={byte1_1[i]*256 + byte2_1[i]}  "
          f"th2={byte3_1[i]*256 + byte4_1[i]}  "
          f"th3={byte5_1[i]*256 + byte6_1[i]}  "
          f"th4={byte7_1[i]*256 + byte8_1[i]}  "
          f"th5={byte9_1[i]*256 + byte10_1[i]}")
    
    if x == 1:
        x = 2
        
    else:
        
        while 1==1:
            response = ser.readline().decode('utf-8').strip()
            if response == "Ready":
                break
        x = 1
    #time.sleep(1)  # pause between steps so servo has time to move

Step  0:  th1=1287  th2=1449  th3=1320  th4=1455  th5=1500
Step  1:  th1=1287  th2=1471  th3=1461  th4=1572  th5=1500


SerialTimeoutException: Write timeout

### path 2: lift

In [174]:
#define arrays to hold the four bytes
byte1_2 = np.zeros(len(path2_full), dtype=int)
byte2_2 = np.zeros(len(path2_full), dtype=int)
byte3_2 = np.zeros(len(path2_full), dtype=int)
byte4_2 = np.zeros(len(path2_full), dtype=int)
byte5_2 = np.zeros(len(path2_full), dtype=int)
byte6_2 = np.zeros(len(path2_full), dtype=int)
byte7_2 = np.zeros(len(path2_full), dtype=int)
byte8_2 = np.zeros(len(path2_full), dtype=int)
byte9_2 = np.zeros(len(path2_full), dtype=int)
byte10_2 = np.zeros(len(path2_full), dtype=int)

#store bytes into temp values to be sent to arduino
for i in range(len(path2_full)):
    byte1_2[i], byte2_2[i] = break_into_two(path2_full[i,0])
    byte3_2[i], byte4_2[i] = break_into_two(path2_full[i,1])
    byte5_2[i], byte6_2[i] = break_into_two(path2_full[i,2])
    byte7_2[i], byte8_2[i] = break_into_two(path2_full[i,3])
    byte9_2[i], byte10_2[i] = break_into_two(path2_full[i,4])

print(byte1_2,'\n\n',byte2_2,'\n\n\n',
      byte3_2,'\n\n',byte4_2,'\n\n\n',
      byte5_2,'\n\n',byte6_2,'\n\n\n',
      byte7_2,'\n\n',byte8_2,'\n\n\n',
      byte9_2,'\n\n',byte10_2)


[5 5] 

 [7 7] 


 [5 5] 

 [177 169] 


 [6 5] 

 [22 40] 


 [6 6] 

 [231   2] 


 [4 4] 

 [26 26]


In [175]:
# Send all path points to both servos
x = 1
for i in range(len(path2_full)):
    WriteByte(ser, int(byte1_2[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2_2[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3_2[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4_2[i]))   # servo 2 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte5_2[i]))   # servo 3 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte6_2[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte7_2[i]))   # servo 4 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte8_2[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte9_2[i]))   # servo 5 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte10_2[i]))   # servo 5 LSB
    time.sleep(0.5)

    print(f"Step {i:2d}:  "
          f"th1={byte1_2[i]*256 + byte2_2[i]}  "
          f"th2={byte3_2[i]*256 + byte4_2[i]}  "
          f"th3={byte5_2[i]*256 + byte6_2[i]}  "
          f"th4={byte7_2[i]*256 + byte8_2[i]}  "
          f"th5={byte9_2[i]*256 + byte10_2[i]}")
    
    if x == 1:
        x = 2
        
    else:
        
        while 1==1:
            response = ser.readline().decode('utf-8').strip()
            if response == "Ready":
                break
        x = 1
    #time.sleep(1)  # pause between steps so servo has time to move

Step  0:  th1=1287  th2=1457  th3=1558  th4=1767  th5=1050
Step  1:  th1=1287  th2=1449  th3=1320  th4=1538  th5=1050


### path 3: under obstacle

In [176]:
#define arrays to hold the four bytes
byte1_3 = np.zeros(len(path3_full), dtype=int)
byte2_3 = np.zeros(len(path3_full), dtype=int)
byte3_3 = np.zeros(len(path3_full), dtype=int)
byte4_3 = np.zeros(len(path3_full), dtype=int)
byte5_3 = np.zeros(len(path3_full), dtype=int)
byte6_3 = np.zeros(len(path3_full), dtype=int)
byte7_3 = np.zeros(len(path3_full), dtype=int)
byte8_3 = np.zeros(len(path3_full), dtype=int)
byte9_3 = np.zeros(len(path3_full), dtype=int)
byte10_3 = np.zeros(len(path3_full), dtype=int)

#store bytes into temp values to be sent to arduino
for i in range(len(path3_full)):
    byte1_3[i], byte2_3[i] = break_into_two(path3_full[i,0])
    byte3_3[i], byte4_3[i] = break_into_two(path3_full[i,1])
    byte5_3[i], byte6_3[i] = break_into_two(path3_full[i,2])
    byte7_3[i], byte8_3[i] = break_into_two(path3_full[i,3])
    byte9_3[i], byte10_3[i] = break_into_two(path3_full[i,4])

print(byte1_3,'\n\n',byte2_3,'\n\n\n',
      byte3_3,'\n\n',byte4_3,'\n\n\n',
      byte5_3,'\n\n',byte6_3,'\n\n\n',
      byte7_3,'\n\n',byte8_3,'\n\n\n',
      byte9_3,'\n\n',byte10_3)

[5 4] 

 [ 7 31] 


 [5 6] 

 [169  58] 


 [5 5] 

 [ 40 174] 


 [6 5] 

 [  2 246] 


 [4 4] 

 [26 26]


In [177]:
# Send all path points to both servos
x = 1
for i in range(len(path3_full)):
    WriteByte(ser, int(byte1_3[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2_3[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3_3[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4_3[i]))   # servo 2 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte5_3[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte6_3[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte7_3[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte8_3[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte9_3[i]))   # servo 5 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte10_3[i]))   # servo 5 LSB
    time.sleep(0.5)

    print(f"Step {i:2d}:  "
          f"th1={byte1_3[i]*256 + byte2_3[i]}  "
          f"th2={byte3_3[i]*256 + byte4_3[i]}  "
          f"th3={byte5_3[i]*256 + byte6_3[i]}  "
          f"th4={byte7_3[i]*256 + byte8_3[i]}  "
          f"th5={byte9_3[i]*256 + byte10_3[i]}")
    
    if x == 1:
        x = 2
        
    else:
        
        while 1==1:
            response = ser.readline().decode('utf-8').strip()
            if response == "Ready":
                break
        x = 1
    #time.sleep(1)  # pause between steps so servo has time to move

Step  0:  th1=1287  th2=1449  th3=1320  th4=1538  th5=1050
Step  1:  th1=1055  th2=1594  th3=1454  th4=1526  th5=1050


### path 4: over to place x

In [178]:
#define arrays to hold the four bytes
byte1_4 = np.zeros(len(path4_full), dtype=int)
byte2_4 = np.zeros(len(path4_full), dtype=int)
byte3_4 = np.zeros(len(path4_full), dtype=int)
byte4_4 = np.zeros(len(path4_full), dtype=int)
byte5_4 = np.zeros(len(path4_full), dtype=int)
byte6_4 = np.zeros(len(path4_full), dtype=int)
byte7_4 = np.zeros(len(path4_full), dtype=int)
byte8_4 = np.zeros(len(path4_full), dtype=int)
byte9_4 = np.zeros(len(path4_full), dtype=int)
byte10_4 = np.zeros(len(path4_full), dtype=int)

#store bytes into temp values to be sent to arduino
for i in range(len(path4_full)):
    byte1_4[i], byte2_4[i] = break_into_two(path4_full[i,0])
    byte3_4[i], byte4_4[i] = break_into_two(path4_full[i,1])
    byte5_4[i], byte6_4[i] = break_into_two(path4_full[i,2])
    byte7_4[i], byte8_4[i] = break_into_two(path4_full[i,3])
    byte9_4[i], byte10_4[i] = break_into_two(path4_full[i,4])

print(byte1_4,'\n\n',byte2_4,'\n\n\n',
      byte3_4,'\n\n',byte4_4,'\n\n\n',
      byte5_4,'\n\n',byte6_4,'\n\n\n',
      byte7_4,'\n\n',byte8_4,'\n\n\n',
      byte9_4,'\n\n',byte10_4)

[4 4 4 4 5 5 6 6 6 7] 

 [ 31  49  80 144  19 193  75 167 226  11] 


 [6 6 6 6 6 6 6 6 6 6] 

 [ 58  91 120 142 148 136 111  80  47  12] 


 [5 5 5 5 5 5 5 5 5 5] 

 [174 193 206 213 215 211 202 187 167 140] 


 [5 5 5 5 5 5 5 5 5 6] 

 [246 232 216 202 197 206 221 237 250   2] 


 [4 4 4 4 4 4 4 4 4 4] 

 [26 26 26 26 26 26 26 26 26 26]


In [179]:
# Send all path points to both servos
x = 1
for i in range(len(path4_full)):
    WriteByte(ser, int(byte1_4[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2_4[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3_4[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4_4[i]))   # servo 2 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte5_4[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte6_4[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte7_4[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte8_4[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte9_4[i]))   # servo 5 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte10_4[i]))   # servo 5 LSB
    time.sleep(0.5)

    print(f"Step {i:2d}:  "
          f"th1={byte1_4[i]*256 + byte2_4[i]}  "
          f"th2={byte3_4[i]*256 + byte4_4[i]}  "
          f"th3={byte5_4[i]*256 + byte6_4[i]}  "
          f"th4={byte7_4[i]*256 + byte8_4[i]}  "
          f"th5={byte9_4[i]*256 + byte10_4[i]}")
    
    if x == 1:
        x = 2
        
    else:
        
        while 1==1:
            response = ser.readline().decode('utf-8').strip()
            if response == "Ready":
                break
        x = 1
    #time.sleep(1)  # pause between steps so servo has time to move

Step  0:  th1=1055  th2=1594  th3=1454  th4=1526  th5=1050
Step  1:  th1=1073  th2=1627  th3=1473  th4=1512  th5=1050
Step  2:  th1=1104  th2=1656  th3=1486  th4=1496  th5=1050
Step  3:  th1=1168  th2=1678  th3=1493  th4=1482  th5=1050
Step  4:  th1=1299  th2=1684  th3=1495  th4=1477  th5=1050
Step  5:  th1=1473  th2=1672  th3=1491  th4=1486  th5=1050
Step  6:  th1=1611  th2=1647  th3=1482  th4=1501  th5=1050
Step  7:  th1=1703  th2=1616  th3=1467  th4=1517  th5=1050
Step  8:  th1=1762  th2=1583  th3=1447  th4=1530  th5=1050
Step  9:  th1=1803  th2=1548  th3=1420  th4=1538  th5=1050


### path 5: up to place y

In [180]:
#define arrays to hold the four bytes
byte1_5 = np.zeros(len(path5_full), dtype=int)
byte2_5 = np.zeros(len(path5_full), dtype=int)
byte3_5 = np.zeros(len(path5_full), dtype=int)
byte4_5 = np.zeros(len(path5_full), dtype=int)
byte5_5 = np.zeros(len(path5_full), dtype=int)
byte6_5 = np.zeros(len(path5_full), dtype=int)
byte7_5 = np.zeros(len(path5_full), dtype=int)
byte8_5 = np.zeros(len(path5_full), dtype=int)
byte9_5 = np.zeros(len(path5_full), dtype=int)
byte10_5 = np.zeros(len(path5_full), dtype=int)

#store bytes into temp values to be sent to arduino
for i in range(len(path5_full)):
    byte1_5[i], byte2_5[i] = break_into_two(path5_full[i,0])
    byte3_5[i], byte4_5[i] = break_into_two(path5_full[i,1])
    byte5_5[i], byte6_5[i] = break_into_two(path5_full[i,2])
    byte7_5[i], byte8_5[i] = break_into_two(path5_full[i,3])
    byte9_5[i], byte10_5[i] = break_into_two(path5_full[i,4])

print(byte1_5,'\n\n',byte2_5,'\n\n\n',
      byte3_5,'\n\n',byte4_5,'\n\n\n',
      byte5_5,'\n\n',byte6_5,'\n\n\n',
      byte7_5,'\n\n',byte8_5,'\n\n\n',
      byte9_5,'\n\n',byte10_5)

[7 6 6] 

 [ 11 155  97] 


 [6 5 5] 

 [ 12 197  77] 


 [5 5 4] 

 [140  73 171] 


 [6 6 5] 

 [  2   6 224] 


 [4 4 4] 

 [26 26 26]


In [181]:
# Send all path points to both servos
x = 1
for i in range(len(path5_full)):
    WriteByte(ser, int(byte1_5[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2_5[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3_5[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4_5[i]))   # servo 2 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte5_5[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte6_5[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte7_5[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte8_5[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte9_5[i]))   # servo 5 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte10_5[i]))   # servo 5 LSB
    time.sleep(0.5)

    print(f"Step {i:2d}:  "
          f"th1={byte1_5[i]*256 + byte2_5[i]}  "
          f"th2={byte3_5[i]*256 + byte4_5[i]}  "
          f"th3={byte5_5[i]*256 + byte6_5[i]}  "
          f"th4={byte7_5[i]*256 + byte8_5[i]}  "
          f"th5={byte9_5[i]*256 + byte10_5[i]}")
    
    if x == 1:
        x = 2
        
    else:
        
        while 1==1:
            response = ser.readline().decode('utf-8').strip()
            if response == "Ready":
                break
        x = 1
    #time.sleep(1)  # pause between steps so servo has time to move

Step  0:  th1=1803  th2=1548  th3=1420  th4=1538  th5=1050
Step  1:  th1=1691  th2=1477  th3=1353  th4=1542  th5=1050
Step  2:  th1=1633  th2=1357  th3=1195  th4=1504  th5=1050


### path 6: down to place z

In [182]:
#define arrays to hold the four bytes
byte1_6 = np.zeros(len(path6_full), dtype=int)
byte2_6 = np.zeros(len(path6_full), dtype=int)
byte3_6 = np.zeros(len(path6_full), dtype=int)
byte4_6 = np.zeros(len(path6_full), dtype=int)
byte5_6 = np.zeros(len(path6_full), dtype=int)
byte6_6 = np.zeros(len(path6_full), dtype=int)
byte7_6 = np.zeros(len(path6_full), dtype=int)
byte8_6 = np.zeros(len(path6_full), dtype=int)
byte9_6 = np.zeros(len(path6_full), dtype=int)
byte10_6 = np.zeros(len(path6_full), dtype=int)

#store bytes into temp values to be sent to arduino
for i in range(len(path6_full)):
    byte1_6[i], byte2_6[i] = break_into_two(path6_full[i,0])
    byte3_6[i], byte4_6[i] = break_into_two(path6_full[i,1])
    byte5_6[i], byte6_6[i] = break_into_two(path6_full[i,2])
    byte7_6[i], byte8_6[i] = break_into_two(path6_full[i,3])
    byte9_6[i], byte10_6[i] = break_into_two(path6_full[i,4])

print(byte1_6,'\n\n',byte2_6,'\n\n\n',
      byte3_6,'\n\n',byte4_6,'\n\n\n',
      byte5_6,'\n\n',byte6_6,'\n\n\n',
      byte7_6,'\n\n',byte8_6,'\n\n\n',
      byte9_6,'\n\n',byte10_6)

[6 6 6] 

 [97 97 97] 


 [5 5 5] 

 [ 77 116  96] 


 [4 5 5] 

 [171 107 212] 


 [5 6 6] 

 [224 121 246] 


 [4 4 4] 

 [26 26 26]


In [183]:
# Send all path points to both servos
x = 1
for i in range(len(path6_full)):
    WriteByte(ser, int(byte1_6[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2_6[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3_6[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4_6[i]))   # servo 2 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte5_6[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte6_6[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte7_6[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte8_6[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte9_6[i]))   # servo 5 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte10_6[i]))   # servo 5 LSB
    time.sleep(0.5)

    print(f"Step {i:2d}:  "
          f"th1={byte1_6[i]*256 + byte2_6[i]}  "
          f"th2={byte3_6[i]*256 + byte4_6[i]}  "
          f"th3={byte5_6[i]*256 + byte6_6[i]}  "
          f"th4={byte7_6[i]*256 + byte8_6[i]}  "
          f"th5={byte9_6[i]*256 + byte10_6[i]}")
    
    if x == 1:
        x = 2
        
    else:
        
        while 1==1:
            response = ser.readline().decode('utf-8').strip()
            if response == "Ready":
                break
        x = 1
    #time.sleep(1)  # pause between steps so servo has time to move

Step  0:  th1=1633  th2=1357  th3=1195  th4=1504  th5=1050
Step  1:  th1=1633  th2=1396  th3=1387  th4=1657  th5=1050
Step  2:  th1=1633  th2=1376  th3=1492  th4=1782  th5=1050


### path 7: raise tip

In [186]:
#define arrays to hold the four bytes
byte1_7 = np.zeros(len(path7_full), dtype=int)
byte2_7 = np.zeros(len(path7_full), dtype=int)
byte3_7 = np.zeros(len(path7_full), dtype=int)
byte4_7 = np.zeros(len(path7_full), dtype=int)
byte5_7 = np.zeros(len(path7_full), dtype=int)
byte6_7 = np.zeros(len(path7_full), dtype=int)
byte7_7 = np.zeros(len(path7_full), dtype=int)
byte8_7 = np.zeros(len(path7_full), dtype=int)
byte9_7 = np.zeros(len(path7_full), dtype=int)
byte10_7 = np.zeros(len(path7_full), dtype=int)

#store bytes into temp values to be sent to arduino
for i in range(len(path7_full)):
    byte1_7[i], byte2_7[i] = break_into_two(path7_full[i,0])
    byte3_7[i], byte4_7[i] = break_into_two(path7_full[i,1])
    byte5_7[i], byte6_7[i] = break_into_two(path7_full[i,2])
    byte7_7[i], byte8_7[i] = break_into_two(path7_full[i,3])
    byte9_7[i], byte10_7[i] = break_into_two(path7_full[i,4])

print(byte1_7,'\n\n',byte2_7,'\n\n\n',
      byte3_7,'\n\n',byte4_7,'\n\n\n',
      byte5_7,'\n\n',byte6_7,'\n\n\n',
      byte7_7,'\n\n',byte8_7,'\n\n\n',
      byte9_7,'\n\n',byte10_7)

[6 6] 

 [97 97] 


 [5 5] 

 [96 77] 


 [5 4] 

 [212 171] 


 [6 5] 

 [246 224] 


 [5 5] 

 [220 220]


In [187]:
# Send all path points to both servos
x = 1
for i in range(len(path7_full)):
    WriteByte(ser, int(byte1_7[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2_7[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3_7[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4_7[i]))   # servo 2 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte5_7[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte6_7[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte7_7[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte8_7[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte9_7[i]))   # servo 5 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte10_7[i]))   # servo 5 LSB
    time.sleep(0.5)

    print(f"Step {i:2d}:  "
          f"th1={byte1_7[i]*256 + byte2_7[i]}  "
          f"th2={byte3_7[i]*256 + byte4_7[i]}  "
          f"th3={byte5_7[i]*256 + byte6_7[i]}  "
          f"th4={byte7_7[i]*256 + byte8_7[i]}  "
          f"th5={byte9_7[i]*256 + byte10_7[i]}")
    
    if x == 1:
        x = 2
        
    else:
        
        while 1==1:
            response = ser.readline().decode('utf-8').strip()
            if response == "Ready":
                break
        x = 1
    #time.sleep(1)  # pause between steps so servo has time to move

Step  0:  th1=1633  th2=1376  th3=1492  th4=1782  th5=1500
Step  1:  th1=1633  th2=1357  th3=1195  th4=1504  th5=1500


### path 8: return to home

In [188]:
#define arrays to hold the four bytes
byte1_8 = np.zeros(len(path8_full), dtype=int)
byte2_8 = np.zeros(len(path8_full), dtype=int)
byte3_8 = np.zeros(len(path8_full), dtype=int)
byte4_8 = np.zeros(len(path8_full), dtype=int)
byte5_8 = np.zeros(len(path8_full), dtype=int)
byte6_8 = np.zeros(len(path8_full), dtype=int)
byte7_8 = np.zeros(len(path8_full), dtype=int)
byte8_8 = np.zeros(len(path8_full), dtype=int)
byte9_8 = np.zeros(len(path8_full), dtype=int)
byte10_8 = np.zeros(len(path8_full), dtype=int)

#store bytes into temp values to be sent to arduino
for i in range(len(path8_full)):
    byte1_8[i], byte2_8[i] = break_into_two(path8_full[i,0])
    byte3_8[i], byte4_8[i] = break_into_two(path8_full[i,1])
    byte5_8[i], byte6_8[i] = break_into_two(path8_full[i,2])
    byte7_8[i], byte8_8[i] = break_into_two(path8_full[i,3])
    byte9_8[i], byte10_8[i] = break_into_two(path8_full[i,4])

print(byte1_8,'\n\n',byte2_8,'\n\n\n',
      byte3_8,'\n\n',byte4_8,'\n\n\n',
      byte5_8,'\n\n',byte6_8,'\n\n\n',
      byte7_8,'\n\n',byte8_8,'\n\n\n',
      byte9_8,'\n\n',byte10_8)

[6 6 5] 

 [ 97  39 220] 


 [5 5 5] 

 [ 77 168 220] 


 [4 5 5] 

 [171 103 220] 


 [5 6 6] 

 [224  65 130] 


 [5 5 5] 

 [220 220 220]


In [189]:
# Send all path points to both servos
x = 1
for i in range(len(path8_full)):
    WriteByte(ser, int(byte1_8[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2_8[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3_8[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4_8[i]))   # servo 2 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte5_8[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte6_8[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte7_8[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte8_8[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte9_8[i]))   # servo 5 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte10_8[i]))   # servo 5 LSB
    time.sleep(0.5)

    print(f"Step {i:2d}:  "
          f"th1={byte1_8[i]*256 + byte2_8[i]}  "
          f"th2={byte3_8[i]*256 + byte4_8[i]}  "
          f"th3={byte5_8[i]*256 + byte6_8[i]}  "
          f"th4={byte7_8[i]*256 + byte8_8[i]}  "
          f"th5={byte9_8[i]*256 + byte10_8[i]}")
    
    if x == 1:
        x = 2
        
    else:
        
        while 1==1:
            response = ser.readline().decode('utf-8').strip()
            if response == "Ready":
                break
        x = 1
    #time.sleep(1)  # pause between steps so servo has time to move

Step  0:  th1=1633  th2=1357  th3=1195  th4=1504  th5=1500
Step  1:  th1=1575  th2=1448  th3=1383  th4=1601  th5=1500
Step  2:  th1=1500  th2=1500  th3=1500  th4=1666  th5=1500


## full path

In [320]:
hold_count = 1
hold4close = np.tile(path2_full[0,:],(hold_count,1))
hold4open = np.tile(path7_full[0,:],(hold_count,1))

path_full = np.vstack([path0_full,  #over pick
                       path1_full,  #pick
                       hold4close,  #hold to ensure gripper closed
                       path2_full,  #lift to safe z
                       path3_full,  #to y under obstacle
                       path4_full,  #to place x
                       path5_full,  #to place y
                       path6_full,  #to place z
                       hold4open,   #hold to ensure gripper opens
                       path7_full,  #retract up
                       path8_full]) #return home

print(path_full)

[[1500.         1500.         1500.         1583.33333333 1500.        ]
 [1287.09320185 1449.03241318 1320.83179444 1455.13271459 1500.        ]
 [1287.09320185 1449.03241318 1320.83179444 1455.13271459 1500.        ]
 [1287.09320185 1471.87269173 1461.3052526  1572.76589421 1500.        ]
 [1287.09320185 1457.87375373 1558.28268454 1683.74226415 1500.        ]
 [1287.09320185 1457.87375373 1558.28268454 1683.74226415 1050.        ]
 [1287.09320185 1457.87375373 1558.28268454 1683.74226415 1050.        ]
 [1287.09320185 1449.03241318 1320.83179444 1455.13271459 1050.        ]
 [1287.09320185 1449.03241318 1320.83179444 1455.13271459 1050.        ]
 [1103.54324019 1581.58255469 1445.6656692  1447.41644784 1050.        ]
 [1103.54324019 1581.58255469 1445.6656692  1447.41644784 1050.        ]
 [1131.67563163 1611.18835891 1464.36111958 1436.50609401 1050.        ]
 [1176.16368378 1636.03540686 1477.37094836 1424.66887483 1050.        ]
 [1248.2902764  1652.9634826  1484.78676899 1415.15

In [321]:
#define arrays to hold the four bytes
byte1 = np.zeros(len(path_full), dtype=int)
byte2 = np.zeros(len(path_full), dtype=int)
byte3 = np.zeros(len(path_full), dtype=int)
byte4 = np.zeros(len(path_full), dtype=int)
byte5 = np.zeros(len(path_full), dtype=int)
byte6 = np.zeros(len(path_full), dtype=int)
byte7 = np.zeros(len(path_full), dtype=int)
byte8 = np.zeros(len(path_full), dtype=int)
byte9 = np.zeros(len(path_full), dtype=int)
byte10 = np.zeros(len(path_full), dtype=int)

#store bytes into temp values to be sent to arduino
for i in range(len(path_full)):
    byte1[i], byte2[i] = break_into_two(path_full[i,0])
    byte3[i], byte4[i] = break_into_two(path_full[i,1])
    byte5[i], byte6[i] = break_into_two(path_full[i,2])
    byte7[i], byte8[i] = break_into_two(path_full[i,3])
    byte9[i], byte10[i] = break_into_two(path_full[i,4])

print(byte1,'\n\n',byte2,'\n\n\n'
      ,byte3,'\n\n',byte4,'\n\n\n'
      ,byte5,'\n\n',byte6,'\n\n\n'
      ,byte7,'\n\n',byte8,'\n\n\n'
      ,byte9,'\n\n',byte10)

[5 5 5 5 5 5 5 5 5 4 4 4 4 4 5 5 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 5] 

 [220   7   7   7   7   7   7   7   7  79  79 107 152 224  75 200  53 135
 194 237 237 147  97  97  97  97  97  97  97  97  97  97  39 220] 


 [5 5 5 5 5 5 5 5 5 6 6 6 6 6 6 6 6 6 6 6 6 5 5 5 5 5 5 5 5 5 5 5 5 5] 

 [220 169 169 191 177 177 177 169 169  45  45  75 100 116 121 112  92  65
  34   1   1 187  77  77 116  96  96  96 115 112  77  77 168 220] 


 [5 5 5 5 6 6 6 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 4 4 5 5 5 5 5 5 4 4 5 5] 

 [220  40  40 181  22  22  22  40  40 165 165 184 197 204 206 202 193 178
 158 131 131  61 171 171 107 212 212 212 149  58 171 171 103 220] 


 [6 5 5 6 6 6 6 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 6 6 6 6 6 5 5 5 5 6] 

 [ 47 175 175  36 147 147 147 175 175 167 167 156 144 135 132 137 148 160
 170 177 177 178 141 141  38 163 163 163  81 249 141 141 238  47] 


 [5 5 5 5 5 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 5 5 5 5 5 5 5 5] 

 [220 220 220 220 220  26  26  26  26  26  26  26  26  26  26  2

In [322]:
# Send all path points to servos
x = 1
for i in range(len(path_full)):
    WriteByte(ser, int(byte1[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4[i]))   # servo 2 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte5[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte6[i]))   # servo 3 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte7[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte8[i]))   # servo 4 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte9[i]))   # servo 5 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte10[i]))   # servo 5 LSB
    time.sleep(0.5) #long pause

    print(f"Step {i:2d}:  "
          f"th1={byte1[i]*256 + byte2[i]}  "
          f"th2={byte3[i]*256 + byte4[i]}  "
          f"th3={byte5[i]*256 + byte6[i]}  "
          f"th4={byte7[i]*256 + byte8[i]}  "
          f"th5={byte9[i]*256 + byte10[i]}")
    
    if x == 1:
        x = 2
        
    else:
        
        while 1==1:
            response = ser.readline().decode('utf-8').strip()
            if response == "Ready":
                break
        x = 1

SerialException: WriteFile failed (PermissionError(13, 'The device does not recognize the command.', None, 22))